##### Citation Info
@InProceedings{Gushchin_2026_CVPR,
    author    = {Gushchin, Aleksandr and Abud, Khaled and Shumitskaya, Ekaterina and Filippov, Artem and Bychkov, Georgii and Lavrushkin, Sergey and Erofeev, Mikhail and Antsiferova, Anastasia and Chen, Changsheng and Tan, Shunquan and Timofte, Radu and Vatolin, Dmitriy and Song, Chuanbiao and Yu, Zijian and Tan, Hao and Lan, Jun and Yang, Zhiqiang and Tang, Yongwei and Wu, Zhiqiang and Seow, Jia Wen and Koay, Hong Vin and Ren, Haodong and Xu, Feng and Chen, Shuai and Xia, Ruiyang and Zhang, Qi and Xu, Yaowen and Zou, Zhaofan and Sun, Hao and Lu, Dagong and Yao, Mufeng and Xu, Xinlei and Wu, Fei and Guo, Fengjun and Luo, Cong and Sharma, Hardik and Negi, Aashish and Shaily, Prateek and Kumar, Jayant and Chaudhary, Sachin and Dudhane, Akshay and Hambarde, Praful and Shukla, Amit and Tu, Zhilin and Li, Fengpeng and Zhang, Jiamin and Fei, Jianwei and Li, Kemou and Wu, Haiwei and Benjdira, Bilel and Ali, Anas M. and Boulila, Wadii and Qu, Chenfan and Li, Junchi},
    title     = {NTIRE 2026 Challenge on Robust AI-Generated Image Detection in the Wild},
    booktitle = {Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR) Workshops},
    month     = {June},
    year      = {2026},
    pages     = {1895-1913}
}


In [2]:
import torch
import torchvision

# check NVIDIA GPU available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
import pandas as pd 
import os
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image 
import torch
class AIGenDetDataset(Dataset):
    def __init__(self, shard_dir, shard_nums=None):
        """ 
        Example dataset class for NTIRE 2026 Robust AI-generated Image Detection In The Wild challenge.

        Arguments:
            - shard_dir (str): base directory where shards are stored 
            - shard_nums (None or list[int]): a list of specific shard numbers (0 to 6) to use. 
                Uses all shards found in shard_dir by default.
        """
        self.shard_root_dir = shard_dir
        if shard_nums is None:
            self.shard_dirs = [os.path.join(shard_dir, f'shard_{i}') for i in range(0,6)]
        else:
            self.shard_dirs = [os.path.join(shard_dir, f'shard_{i}') for i in shard_nums]
        self.shard_dirs = [x for x in self.shard_dirs if os.path.isdir(x)]
        label_dfs = [pd.read_csv(os.path.join(x, 'labels.csv'), index_col=0) for x in self.shard_dirs]
        self.label_df = pd.DataFrame(columns=['image_name', 'label', 'shard_name'])
        for idx, ldf in enumerate(label_dfs):
            ldf['shard_name'] = Path(self.shard_dirs[idx]).name
            self.label_df = pd.concat([self.label_df, ldf], ignore_index=True)
        print(f'Found {len(self.shard_dirs)} shards, {len(self.label_df)} images in total.')

    def __len__(self):
        return len(self.label_df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_path = os.path.join(self.shard_root_dir,
                                self.label_df.loc[idx, 'shard_name'], 'images', self.label_df.loc[idx, 'image_name'])
        image = Image.open(img_path)
        label = self.label_df.loc[idx, 'label']
        sample = {'image': image, 'label': label}
        return sample


In [4]:
# Set up image transforms, datasets, and data loaders

from pathlib import Path
from datasets import load_dataset
from torch.utils.data import DataLoader

from preprocessing import preprocess

batch_size = 32

path = "../data" # path where dataset is stored
train_shard_nums = [0] # shard numbers to use (0-5) -- using one for ease right now, set 5 aside for val.
train = AIGenDetDataset(path, train_shard_nums)
val = AIGenDetDataset(path, [5])

trainloader = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
valloader = DataLoader(val, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)


c:\Users\butch\BDAA-Human-AI-Image-Detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: '../data\\shard_0\\labels.csv'

In [ ]:
# Set up model for transfer learning

import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

model = resnet50(weights=ResNet50_Weights.DEFAULT)

# Replace the final layer with a new one that has 2 output classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

for name, parameter in model.named_parameters():
    parameter.requires_grad = name.startswith(
        ("layer3.", "layer4.", "fc.")
    )

model.to(device)


In [ ]:
# set up optimizer

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {"params": model.layer3.parameters(), "lr": 1e-5},
    {"params": model.layer4.parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 1e-4},
])

In [ ]:
# Train/eval loop
from sklearn.metrics import f1_score

best_f1 = float('-inf')
miss_count = 0
max_misses = 5

for epoch in range(50):
    model.train()
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


    running_loss = 0.0

    for i, batch in enumerate(trainloader, 0):
        images = batch["Image"].to(device)
        binary_labels = batch["Label_A"].to(device)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, binary_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:    # print every 200 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.6f}')
            running_loss = 0.0
    print(f'Finished epoch {epoch + 1}')
    
    # Validation
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in valloader:
            images = batch["Image"].to(device)
            predictions = model(images).argmax(dim=1)

            y_true.extend(batch["Label_A"].tolist())
            y_pred.extend(predictions.tolist())

    val_f1 = f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    )
    print(f"Validation weighted F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        miss_count = 0
        torch.save(model, "./models/ResNet50FineTuned.pth")
    else:
        miss_count += 1
        if miss_count >= max_misses:
            print("Weighted F1 stopped improving, stopping early.")
            break

print('Finished Training')

In [ ]:
# Get detailed metrics for the current model on the validation set
model = torch.load('models/ResNet50FineTuned.pth', map_location=device, weights_only=False)

import matplotlib.pyplot as plt
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

y_true, y_pred, y_ai_probability = [], [], []

model.eval()
with torch.no_grad():
    for batch in valloader:
        images = batch["Image"].to(device)
        outputs = model(images)
        probabilities = outputs.softmax(dim=1)
        predictions = probabilities.argmax(dim=1)

        y_true.extend(batch["Label_A"].to(device).tolist())
        y_pred.extend(predictions.tolist())
        # Class 1 is AI-generated; use its probability for every image.
        y_ai_probability.extend(probabilities[:, 1].tolist())

print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Real", "AI-generated"],
    digits=3,
    zero_division=0,
))

print("Confusion matrix (rows: actual, columns: predicted; order: Real, AI-generated):")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
print(f"Brier score (lower is better): {brier_score_loss(y_true, y_ai_probability):.4f}")
print(f"Log loss (lower is better): {log_loss(y_true, y_ai_probability, labels=[0, 1]):.4f}")
print(f"ROC-AUC (higher is better): {roc_auc_score(y_true, y_ai_probability):.4f}")

# Near the diagonal means predicted AI probabilities match observed AI frequencies.
CalibrationDisplay.from_predictions(
    y_true,
    y_ai_probability,
    n_bins=10,
    strategy="uniform",
    name="ResNet50 tuned",
)
plt.title("Validation calibration: AI-generated probability")
plt.tight_layout()
plt.show()
